<a href="https://colab.research.google.com/github/vorhersager/deep-learning-jax/blob/main/Tutorial_01_Mathematical_Foundations_in_JAX_and_TensorFlow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Tutorial 1: Mathematical Foundations in JAX and TensorFlow


Instructor: John Sipple

**Topic: Computational Linear Algebra, Information Theory Measures, & Optimization Primitives**

### Overview

In this tutorial, we will establish the computational groundwork required for understanding modern neural architectures. Rather than relying entirely on high-level APIs, we will manually implement foundational mathematical operations to compare the paradigms of two dominant machine learning frameworks:

* **TensorFlow**: An established, ecosystem-rich framework utilizing dynamic computational graphs (Eager execution) and robust, pre-packaged loss functions.


* **JAX**: A functional, composable transformation framework that brings Autograd and Accelerated Linear Algebra (XLA) directly to Python and NumPy, exposing low-level math more transparently.



This notebook serves as a bridge from analytical mathematics to iterative deep learning. It explores how these frameworks handle numerical stability, system dynamics, probability distributions, and the complex tensor operations essential for building state-of-the-art models like Transformers and GANs.

### Key Concepts Explored:

* **Computational Linear Algebra (Matrix Inversion):** Understanding numerical stability and explicit inversion for analytical solutions and second-order optimization.


* **Automatic Differentiation & The Jacobian:** Moving beyond simple scalar gradients to compute the full Jacobian matrix using functional transformations (`jacfwd`, `jacrev`) and `GradientTape` to map input-output sensitivities.


* **Information Theory Measures:** Manually implementing Cross-Entropy and Kullback-Leibler (KL) Divergence to measure the "distance" between probability distributions.


* **Singular Value Decomposition (SVD):** Factorizing non-square matrices to understand techniques like low-rank approximation and Spectral Normalization used to stabilize deep ResNets and GANs.


* **Optimization Primitives:** Calculating gradients and manually applying a Gradient Descent step to a simple quadratic loss function.


* **High-Dimensional Tensor Products (Einsum):** Utilizing Einstein Summation index notation to cleanly execute complex operations like Batch Matrix Multiplication, a critical skill for working with 4D tensors in Transformer models.

In [ ]:
# Install JAX if necessary (Colab usually includes it, but good practice to check)
!pip install --upgrade -q jax jaxlib tensorflow

import tensorflow as tf
import jax
import jax.numpy as jnp
import numpy as np

# Set JAX to use 64-bit precision for strict comparisons (optional, usually 32-bit in DL)
jax.config.update("jax_enable_x64", True)

print(f"TensorFlow Version: {tf.__version__}")
print(f"JAX Version: {jax.__version__}")
print(f"JAX Devices: {jax.devices()}")

## Part 1: Linear Algebra - Matrix Inversion

While explicit matrix inversion is rare in training deep networks (due to computational cost $O(n^3)$), it is fundamental in analytical solutions (e.g., Normal Equation) and second-order optimization methods.We will compute $A^{-1}$ such that $A \cdot A^{-1} = I$.

In [ ]:
# --- Setup Data ---
# Create a random 3x3 matrix
key = jax.random.PRNGKey(2026)
shape = (3, 3)

# Generate generic data in NumPy to share across frameworks
np_A = np.random.randn(*shape)

# Ensure it is invertible by adding identity (dominating diagonal)
np_A = np_A + np.eye(shape[0]) * 2

print("Matrix A:\n", np_A)

# ---------------------------------------------------------
# 1. TensorFlow Implementation
# ---------------------------------------------------------
tf_A = tf.constant(np_A)
tf_inv = tf.linalg.inv(tf_A)

# Verification: A * A_inv should be Identity
tf_check = tf.matmul(tf_A, tf_inv)

print("\n--- TensorFlow Results ---")
print("Inverse:\n", tf_inv.numpy())
print("Check (A * A_inv):\n", tf_check.numpy())

# ---------------------------------------------------------
# 2. JAX Implementation
# ---------------------------------------------------------
jax_A = jnp.array(np_A)
jax_inv = jnp.linalg.inv(jax_A)

# Verification
jax_check = jnp.dot(jax_A, jax_inv)

print("\n--- JAX Results ---")
print("Inverse:\n", jax_inv)
print("Check (A * A_inv):\n", jax_check)

## Part 2: Automatic Differentiation - The Jacobian

In Deep Learning, we typically calculate the **Gradient** (vector of partial derivatives of a scalar loss with respect to weights). However, for understanding system dynamics, complex mappings, or generative models (like GANs or VAEs), we often need the Jacobian.Given a function $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$, the Jacobian $J$ is an $m \times n$ matrix where:$$J_{ij} = \frac{\partial f_i}{\partial x_j}$$Let's define a simple vector-valued function: $f(\mathbf{x}) = [\sin(x_1), \cos(x_1) \cdot x_2]$

In [ ]:
# Define the function mathematically
def my_mapping(x):
    # x is a vector of size 2
    # Returns a vector of size 2
    y1 = jnp.sin(x[0]) if isinstance(x, jax.Array) else tf.sin(x[0])
    y2 = jnp.cos(x[0]) * x[1] if isinstance(x, jax.Array) else tf.cos(x[0]) * x[1]

    if isinstance(x, jax.Array):
        return jnp.stack([y1, y2])
    else:
        return tf.stack([y1, y2])

# Input point
x_val = np.array([0.5, 2.0])

# ---------------------------------------------------------
# 1. TensorFlow Implementation (GradientTape)
# ---------------------------------------------------------
x_tf = tf.Variable(x_val, dtype=tf.float64)

with tf.GradientTape() as tape:
    y_tf = my_mapping(x_tf)

# Jacobian: TF handles the batch dimensions automatically
J_tf = tape.jacobian(y_tf, x_tf)

print("--- TensorFlow Jacobian ---")
print(J_tf.numpy())

# ---------------------------------------------------------
# 2. JAX Implementation (Functional Transforms)
# ---------------------------------------------------------
# JAX requires explicit transformations.
# jax.jacfwd uses forward-mode autodiff (efficient for tall matrices n << m)
# jax.jacrev uses reverse-mode autodiff (efficient for wide matrices n >> m)

x_jax = jnp.array(x_val, dtype=jnp.float64)
J_jax = jax.jacfwd(my_mapping)(x_jax)

print("\n--- JAX Jacobian ---")
print(J_jax)

## Part 3: Information Theory Measures

In classification tasks, we rarely use Mean Squared Error. We measure the "distance" between probability distributions.

###3.1 Cross-Entropy
Used primarily as a loss function for classification.$$H(p, q) = - \sum_{x} p(x) \log q(x)$$Where $p$ is the true distribution (usually one-hot) and $q$ is the predicted distribution.

###3.2 KL Divergence (Kullback–Leibler)

Used in Variational Autoencoders (VAEs) and distillation. It measures how much information is lost when approximating $p$ with $q$.$$D_{KL}(P || Q) = \sum_{x} P(x) \log \left( \frac{P(x)}{Q(x)} \right)$$Note: In the examples below, we assume inputs are probabilities (softmax has already been applied).

In [ ]:
# --- Setup Data ---
# True distribution (P): e.g., a one-hot label smoothed slightly
P_np = np.array([0.1, 0.8, 0.1])
# Predicted distribution (Q): output of a softmax
Q_np = np.array([0.2, 0.5, 0.3])

# ---------------------------------------------------------
# 1. TensorFlow Implementation (Built-in Losses)
# ---------------------------------------------------------
# NOTE: TF losses usually expect batches. We add a batch dim [1, 3]
P_tf = tf.constant([P_np])
Q_tf = tf.constant([Q_np])

# Cross Entropy
cce_func = tf.keras.losses.CategoricalCrossentropy(from_logits=False)
cce_val = cce_func(P_tf, Q_tf)

# KL Divergence
kld_func = tf.keras.losses.KLDivergence()
kld_val = kld_func(P_tf, Q_tf)

print("--- TensorFlow Metrics ---")
print(f"Cross Entropy: {cce_val.numpy():.4f}")
print(f"KL Divergence: {kld_val.numpy():.4f}")

# ---------------------------------------------------------
# 2. JAX Implementation (Manual & Optax)
# ---------------------------------------------------------
# While libraries like 'optax' exist, writing these from scratch
# in JAX builds intuition.

P_jax = jnp.array(P_np)
Q_jax = jnp.array(Q_np)

# Manual Cross Entropy
# Add a small epsilon for numerical stability inside log
def cross_entropy(p, q):
    return -jnp.sum(p * jnp.log(q + 1e-9))

# Manual KL Divergence
# D_KL = sum( p * (log(p) - log(q)) )
def kl_divergence(p, q):
    return jnp.sum(p * (jnp.log(p + 1e-9) - jnp.log(q + 1e-9)))

jax_cce = cross_entropy(P_jax, Q_jax)
jax_kld = kl_divergence(P_jax, Q_jax)

print("\n--- JAX Metrics ---")
print(f"Cross Entropy: {jax_cce:.4f}")
print(f"KL Divergence: {jax_kld:.4f}")

## Summary

* **Linear Algebra**: JAX mimics NumPy closely (jax.numpy), while TensorFlow uses tf.linalg.

* **Jacobians**: TF uses the context manager GradientTape. JAX uses functional transformations (jacfwd, jacrev) which can be composed arbitrarily.

* **Information Theory**: Both frameworks handle these computations efficiently, but JAX exposes the low-level math more transparently, whereas TF provides robust, pre-packaged Keras loss functions.

## Part 4: Advanced Challenges
The following concepts are critical for understanding modern Deep Learning architectures (like Transformers) and training stability.



###4.1 Singular Value Decomposition (SVD)
SVD is the factorization of a real or complex matrix. It is crucial for:

1. Low-rank approximation: Compressing large weight matrices.
3. Spectral Normalization: Stabilizing GANs and deep ResNets by limiting the largest singular value.

Given $A$, we compute $U, S, V^T$ such that:$$A = U \Sigma V^T$$Where $S$ (or $\Sigma$) contains the singular values (non-negative, descending magnitude).

In [ ]:
# --- Setup Data ---
# A non-square matrix (e.g., a weight matrix transforming 4 features to 3)
A_np = np.random.randn(3, 4)

print(f"Original Matrix Shape: {A_np.shape}")

# ---------------------------------------------------------
# 1. TensorFlow SVD
# ---------------------------------------------------------
A_tf = tf.constant(A_np)

# compute_uv=True is required to get the singular vectors
s_tf, u_tf, v_tf = tf.linalg.svd(A_tf)

# Reconstruction: U * diag(S) * V_transpose
# Note: TF returns V, not V^T, so we must transpose it for reconstruction
# We must also cast S into a diagonal matrix
A_reconst_tf = tf.matmul(u_tf, tf.matmul(tf.linalg.diag(s_tf), v_tf, adjoint_b=True))

print("\n--- TensorFlow SVD ---")
print(f"Singular Values: {s_tf.numpy()}")
print(f"Reconstruction Error: {tf.norm(A_tf - A_reconst_tf).numpy():.6e}")

# ---------------------------------------------------------
# 2. JAX SVD
# ---------------------------------------------------------
A_jax = jnp.array(A_np)

# JAX usually returns U, S, V_T (V is already transposed in some implementations,
# but numpy.linalg.svd returns V^T. Let's verify JAX behavior:
# JAX matches NumPy: it returns U, S, Vh (Hermitian/Transpose of V)
u_jax, s_jax, vh_jax = jnp.linalg.svd(A_jax, full_matrices=False)

# Reconstruction
A_reconst_jax = jnp.dot(u_jax, jnp.dot(jnp.diag(s_jax), vh_jax))

print("\n--- JAX SVD ---")
print(f"Singular Values: {s_jax}")
print(f"Reconstruction Error: {jnp.linalg.norm(A_jax - A_reconst_jax):.6e}")

###4.2 Gradients & Optimization Steps

While the **Jacobian** gives us the full derivative map, in training, we usually want the Gradient of a scalar loss function with respect to input parameters.Challenge: Implement a single Gradient Descent step manually.$$\theta_{new} = \theta_{old} - \eta \cdot \nabla_{\theta} L(\theta)$$We will use a simple quadratic bowl loss: $L(\theta) = \frac{1}{2} ||\theta||^2$.

In [ ]:
# --- Setup ---
theta_start = np.array([3.0, 4.0]) # Starting at x=3, y=4
learning_rate = 0.1

# ---------------------------------------------------------
# 1. TensorFlow Gradient Step
# ---------------------------------------------------------
theta_tf = tf.Variable(theta_start)

with tf.GradientTape() as tape:
    # Loss: 0.5 * (x^2 + y^2)
    loss = 0.5 * tf.reduce_sum(tf.square(theta_tf))

# Calculate Gradient
grads = tape.gradient(loss, theta_tf)

# Apply Update
theta_tf.assign_sub(learning_rate * grads)

print("--- TensorFlow Optimization ---")
print(f"Gradient: {grads.numpy()}")
print(f"Updated Theta: {theta_tf.numpy()}")

# ---------------------------------------------------------
# 2. JAX Gradient Step (Functional)
# ---------------------------------------------------------
# Define the loss function
def loss_fn(theta):
    return 0.5 * jnp.sum(theta ** 2)

theta_jax = jnp.array(theta_start)

# jax.grad returns a function that computes the gradient
grad_fn = jax.grad(loss_fn)
gradients_jax = grad_fn(theta_jax)

# JAX arrays are immutable. We return a NEW array for the update.
theta_updated_jax = theta_jax - learning_rate * gradients_jax

print("\n--- JAX Optimization ---")
print(f"Gradient: {gradients_jax}")
print(f"Updated Theta: {theta_updated_jax}")

##4.3 High-Dimensional Tensor Products (Einsum)

In Transformer models (like GPT/BERT), we deal with 4D tensors: [Batch, Heads, Sequence, features]. Standard matrix multiplication (matmul) can become confusing with broadcasting rules.


**Einstein Summation (einsum)** allows us to describe operations using index notation. Example: **Batch Matrix Multiplication**.

Given inputs:

* $A$ shape: (Batch, N, K)
* $B$ shape: (Batch, K, M)
* Output shape: (Batch, N, M)

Equation: $C_{bnm} = \sum_k A_{bnk} B_{bkm}$

Einsum String: `'bnk,bkm->bnm'`

In [ ]:
# --- Setup Data ---
# Batch size=2, N=3, K=4, M=5
batch_A = np.random.randn(2, 3, 4)
batch_B = np.random.randn(2, 4, 5)

# ---------------------------------------------------------
# 1. TensorFlow Einsum
# ---------------------------------------------------------
res_tf = tf.einsum('bnk,bkm->bnm', tf.constant(batch_A), tf.constant(batch_B))

print("--- TensorFlow Einsum ---")
print(f"Output Shape: {res_tf.shape}")
# Verify first batch item matches standard matmul
print(f"Check Batch 0:\n {res_tf[0, :2, :2].numpy()}")

# ---------------------------------------------------------
# 2. JAX Einsum
# ---------------------------------------------------------
# JAX's syntax is identical for einsum, which is part of its power
res_jax = jnp.einsum('bnk,bkm->bnm', jnp.array(batch_A), jnp.array(batch_B))

print("\n--- JAX Einsum ---")
print(f"Output Shape: {res_jax.shape}")
print(f"Check Batch 0:\n {res_jax[0, :2, :2]}")

# Final Conclusion

You have now implemented the core mathematical building blocks of deep learning in both TensorFlow and JAX:

1. **Inversion**: For analytical stability.

2. **Jacobians**: For understanding input-output sensitivity.

3. **Divergences**: The "rulers" we use to measure learning progress.

4. **SVD**: For factorizing and stabilizing matrices.

5. **Gradients**: The engine of learning.

6. **Einsum**: The language of high-dimensional tensor operations (Transformers).

